# 🔄 Customer Churn Prediction + Explainability Dashboard
> **Resume-worthy DS/ML Project** | Built for Google Colab

---

## 📌 Project Overview
- **Problem**: Predict which telecom customers will churn (cancel service) before they do
- **Goal**: Build an end-to-end ML pipeline with EDA → Feature Engineering → Modeling → Explainability
- **Dataset**: IBM Telco Customer Churn (via Kaggle / direct URL)
- **Output**: Trained model + SHAP explainability plots + interactive prediction widget

## 🗺️ Notebook Structure
| Section | Description |
|---|---|
| 1 | Setup & Data Loading |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Feature Engineering & Preprocessing |
| 4 | Model Training & Evaluation |
| 5 | SHAP Explainability |
| 6 | Interactive Prediction Widget |
| 7 | Business Insights & Recommendations |

---
# ⚙️ Section 1: Setup & Data Loading

In [ ]:
# ── Install required libraries ──────────────────────────────────────────────
!pip install shap xgboost imbalanced-learn --quiet
print('✅ All libraries installed successfully!')

In [ ]:
# ── Core Imports ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, precision_recall_curve,
                              ConfusionMatrixDisplay, f1_score, accuracy_score)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

# Explainability
import shap
shap.initjs()

# Interactive Widgets
import ipywidgets as widgets
from IPython.display import display, HTML

# Plot styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
PALETTE = ['#2563eb', '#ef4444', '#10b981', '#f59e0b', '#8b5cf6']
sns.set_palette(PALETTE)

print('✅ All imports successful!')

In [ ]:
# ── Load Dataset ─────────────────────────────────────────────────────────────
# Option A: Load directly from URL (no Kaggle account needed)
URL = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'

try:
    df = pd.read_csv(URL)
    print(f'✅ Dataset loaded from URL')
except:
    # Option B: Upload manually
    # If URL fails, run: from google.colab import files; uploaded = files.upload()
    # Then: df = pd.read_csv(list(uploaded.keys())[0])
    print('❌ URL failed. Please upload the CSV manually using the cell below.')

print(f'\n📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

In [ ]:
# ── (OPTIONAL) Manual Upload from local machine ──────────────────────────────
# Uncomment and run this cell ONLY if the URL above failed

# from google.colab import files
# print('Click "Choose Files" and upload WA_Fn-UseC_-Telco-Customer-Churn.csv')
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])
# print(f'✅ Uploaded: {df.shape}')

---
# 🔍 Section 2: Exploratory Data Analysis (EDA)
> Understand the data before touching the model

In [ ]:
# ── 2.1 Basic Data Overview ──────────────────────────────────────────────────
print('=' * 55)
print('           DATASET OVERVIEW')
print('=' * 55)
print(f'  Rows          : {df.shape[0]:,}')
print(f'  Columns       : {df.shape[1]}')
print(f'  Memory Usage  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
print()
print('  Column Data Types:')
print(df.dtypes.value_counts().to_string())
print()

# Missing values
missing = df.isnull().sum()
if missing.sum() == 0:
    print('  Missing Values: ✅ None found')
else:
    print('  Missing Values:')
    print(missing[missing > 0])

print()
print('  Churn Distribution:')
churn_counts = df['Churn'].value_counts()
churn_pct    = df['Churn'].value_counts(normalize=True) * 100
for k in churn_counts.index:
    print(f'    {k:4s} : {churn_counts[k]:,} ({churn_pct[k]:.1f}%)')
print('=' * 55)

In [ ]:
# ── 2.2 Fix TotalCharges & Encode Target ─────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)
print('✅ TotalCharges converted to numeric')
print('✅ Churn encoded as binary (1=Yes, 0=No)')

In [ ]:
# ── 2.3 Churn Rate Overview Plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Customer Churn — High-Level Overview', fontsize=14, fontweight='bold', y=1.02)

# Plot 1: Churn distribution (donut)
ax = axes[0]
sizes  = df['Churn'].value_counts()
colors = ['#2563eb', '#ef4444']
wedges, texts, autotexts = ax.pie(
    sizes, labels=sizes.index, autopct='%1.1f%%',
    colors=colors, startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for at in autotexts: at.set_fontsize(12); at.set_fontweight('bold')
ax.set_title('Overall Churn Rate', fontweight='bold')

# Plot 2: Churn by Contract Type
ax = axes[1]
contract_churn = df.groupby('Contract')['Churn_Binary'].mean() * 100
bars = ax.bar(contract_churn.index, contract_churn.values,
              color=['#ef4444','#f59e0b','#10b981'], edgecolor='white', linewidth=1.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Churn Rate by Contract Type', fontweight='bold')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')

# Plot 3: Churn by Tenure Bucket
ax = axes[2]
df['tenure_bucket'] = pd.cut(df['tenure'], bins=[0,12,24,48,72],
                              labels=['0-12m','13-24m','25-48m','49-72m'])
tenure_churn = df.groupby('tenure_bucket', observed=True)['Churn_Binary'].mean() * 100
bars = ax.bar(tenure_churn.index, tenure_churn.values,
              color=PALETTE[:4], edgecolor='white', linewidth=1.5)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Churn Rate by Customer Tenure', fontweight='bold')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Churn Rate (%)')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('churn_overview.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n💡 Key Insight: Month-to-month customers churn at 3-4x the rate of long-term contract holders!')

In [ ]:
# ── 2.4 Numeric Features vs Churn ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Numeric Features: Churned vs Retained Customers', fontsize=14, fontweight='bold')

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
colors_map = {0: '#2563eb', 1: '#ef4444'}
labels_map = {0: 'Retained', 1: 'Churned'}

for ax, col in zip(axes, num_cols):
    for churn_val in [0, 1]:
        subset = df[df['Churn_Binary'] == churn_val][col]
        ax.hist(subset, bins=35, alpha=0.65, color=colors_map[churn_val],
                label=labels_map[churn_val], edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig('numeric_features.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n💡 Key Insight: Churned customers have LOWER tenure and HIGHER monthly charges on average!')

In [ ]:
# ── 2.5 Categorical Features Churn Heatmap ───────────────────────────────────
cat_cols = ['gender','SeniorCitizen','Partner','Dependents',
            'PhoneService','MultipleLines','InternetService',
            'OnlineSecurity','TechSupport','StreamingTV',
            'PaperlessBilling','PaymentMethod']

cat_churn_df = pd.DataFrame({
    col: df.groupby(col)['Churn_Binary'].mean() * 100
    for col in cat_cols
    if df[col].nunique() <= 4
}).T.round(1)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(cat_churn_df, annot=True, fmt='.1f', cmap='RdYlGn_r',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Churn Rate (%)'})
ax.set_title('Churn Rate (%) by Categorical Features', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('')
plt.tight_layout()
plt.savefig('categorical_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n💡 Key Insight: Customers without OnlineSecurity and TechSupport churn significantly more!')

---
# 🛠️ Section 3: Feature Engineering & Preprocessing

In [ ]:
# ── 3.1 Feature Engineering ──────────────────────────────────────────────────
df_model = df.copy()

# Feature 1: Average monthly spend proxy
df_model['AvgMonthlySpend'] = np.where(
    df_model['tenure'] > 0,
    df_model['TotalCharges'] / df_model['tenure'],
    df_model['MonthlyCharges']
)

# Feature 2: Number of services subscribed
service_cols = ['PhoneService','MultipleLines','OnlineSecurity',
                'OnlineBackup','DeviceProtection','TechSupport',
                'StreamingTV','StreamingMovies']
df_model['ServiceCount'] = df_model[service_cols].apply(
    lambda row: sum(1 for v in row if v == 'Yes'), axis=1
)

# Feature 3: Has no support services (risk flag)
df_model['NoSupportServices'] = (
    (df_model['OnlineSecurity'] == 'No') &
    (df_model['TechSupport'] == 'No')
).astype(int)

# Feature 4: Is on a monthly contract (high-risk flag)
df_model['IsMonthToMonth'] = (df_model['Contract'] == 'Month-to-month').astype(int)

# Feature 5: Tenure group (early, mid, long)
df_model['TenureGroup'] = pd.cut(df_model['tenure'],
                                  bins=[0, 12, 36, 72],
                                  labels=[0, 1, 2]).astype(int)

print('✅ New features created:')
new_feats = ['AvgMonthlySpend','ServiceCount','NoSupportServices','IsMonthToMonth','TenureGroup']
print(df_model[new_feats].describe().T[['mean','min','max']].round(2))

In [ ]:
# ── 3.2 Encode Categorical Variables ─────────────────────────────────────────
# Drop customerID (not predictive)
df_model.drop(columns=['customerID','Churn','tenure_bucket'], inplace=True)

# Binary map for 2-category columns
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
binary_cols = ['gender','Partner','Dependents','PhoneService',
               'PaperlessBilling','SeniorCitizen']

for col in binary_cols:
    if df_model[col].dtype == object:
        df_model[col] = df_model[col].map(binary_map).fillna(df_model[col])

# One-hot encode remaining categorical columns
cat_remaining = df_model.select_dtypes(include='object').columns.tolist()
cat_remaining = [c for c in cat_remaining if c != 'Churn_Binary']
df_model = pd.get_dummies(df_model, columns=cat_remaining, drop_first=True)

# Ensure all booleans → int
bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

print(f'✅ Encoding complete. Final shape: {df_model.shape}')
print(f'   Features: {df_model.shape[1]-1}')

In [ ]:
# ── 3.3 Train/Test Split + SMOTE ─────────────────────────────────────────────
X = df_model.drop(columns=['Churn_Binary'])
y = df_model['Churn_Binary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features
scaler = StandardScaler()
num_features = ['tenure','MonthlyCharges','TotalCharges','AvgMonthlySpend','ServiceCount']
X_train[num_features] = scaler.fit_transform(X_train[num_features])
X_test[num_features]  = scaler.transform(X_test[num_features])

# Apply SMOTE to handle class imbalance on training set only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('✅ Train/Test Split:')
print(f'   Train (before SMOTE): {X_train.shape[0]:,} rows')
print(f'   Train (after SMOTE) : {X_train_sm.shape[0]:,} rows  ← balanced!')
print(f'   Test                : {X_test.shape[0]:,} rows')
print(f'\n   Class balance after SMOTE: {pd.Series(y_train_sm).value_counts().to_dict()}')

---
# 🤖 Section 4: Model Training & Evaluation

In [ ]:
# ── 4.1 Train Multiple Models ─────────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=8,
                                                   random_state=42, n_jobs=-1),
    'XGBoost'            : XGBClassifier(n_estimators=200, learning_rate=0.05,
                                          max_depth=5, use_label_encoder=False,
                                          eval_metric='logloss', random_state=42,
                                          verbosity=0)
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Training models...\n')
for name, model in models.items():
    model.fit(X_train_sm, y_train_sm)
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    cv_auc  = cross_val_score(model, X_train_sm, y_train_sm,
                               cv=cv, scoring='roc_auc', n_jobs=-1).mean()
    results[name] = {
        'model'   : model,
        'y_pred'  : y_pred,
        'y_proba' : y_proba,
        'Accuracy': accuracy_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'ROC-AUC' : roc_auc_score(y_test, y_proba),
        'CV AUC'  : cv_auc
    }
    print(f'  ✅ {name:<25} | AUC={roc_auc_score(y_test, y_proba):.4f} | F1={f1_score(y_test, y_pred):.4f}')

print('\n✅ All models trained!')

In [ ]:
# ── 4.2 Model Comparison Table ───────────────────────────────────────────────
comparison_df = pd.DataFrame({
    name: {k: v for k, v in res.items() if k not in ('model','y_pred','y_proba')}
    for name, res in results.items()
}).T.round(4)

# Highlight best in each column
display(HTML('<h3 style="font-family:sans-serif">📊 Model Comparison</h3>'))
display(comparison_df.style
    .format('{:.4f}')
    .highlight_max(color='#bbf7d0', axis=0)
    .set_table_styles([{'selector':'th','props':[('background','#1e3a5f'),('color','white'),('font-size','12px')]}])
    .set_caption('Green = Best per metric')
)

best_model_name = comparison_df['ROC-AUC'].idxmax()
best_model      = results[best_model_name]['model']
print(f'\n🏆 Best Model: {best_model_name} (ROC-AUC = {comparison_df["ROC-AUC"].max():.4f})')

In [ ]:
# ── 4.3 ROC Curves + Confusion Matrix ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

# ROC Curves
ax = axes[0]
for (name, res), color in zip(results.items(), PALETTE):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    auc = res['ROC-AUC']
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'--', color='gray', lw=1, label='Random Baseline')
ax.fill_between([0,1],[0,1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend(fontsize=9)

# Confusion Matrix for best model
ax = axes[1]
cm = confusion_matrix(y_test, results[best_model_name]['y_pred'])
disp = ConfusionMatrixDisplay(cm, display_labels=['Retained','Churned'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — {best_model_name}', fontweight='bold')

plt.tight_layout()
plt.savefig('model_evaluation.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 4.4 Classification Report for Best Model ─────────────────────────────────
print(f'Classification Report — {best_model_name}')
print('='*55)
print(classification_report(y_test, results[best_model_name]['y_pred'],
                             target_names=['Retained','Churned']))

In [ ]:
# ── 4.5 Threshold Optimisation (Business View) ───────────────────────────────
# Adjust threshold to maximise F1 (can tune for business cost/benefit)
y_proba_best = results[best_model_name]['y_proba']
thresholds   = np.arange(0.1, 0.9, 0.01)
f1_scores    = [f1_score(y_test, (y_proba_best >= t).astype(int)) for t in thresholds]
best_thresh  = thresholds[np.argmax(f1_scores)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_scores, color='#2563eb', lw=2)
ax.axvline(x=best_thresh, color='#ef4444', linestyle='--', lw=1.5,
           label=f'Optimal threshold = {best_thresh:.2f}')
ax.axvline(x=0.5, color='gray', linestyle=':', lw=1.5, label='Default (0.5)')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('F1 Score')
ax.set_title('Threshold vs F1 Score — Find the Best Decision Boundary', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('threshold_tuning.png', bbox_inches='tight', dpi=150)
plt.show()
print(f'\n💡 Optimal Threshold: {best_thresh:.2f} → F1 = {max(f1_scores):.4f}')
print(f'   (Default 0.5 threshold F1 = {f1_score(y_test, (y_proba_best>=0.5).astype(int)):.4f})')

---
# 🔎 Section 5: SHAP Explainability
> *"Why did the model predict churn for this customer?"*
> This is the most interview-impressive part of the project.

In [ ]:
# ── 5.1 Compute SHAP Values ──────────────────────────────────────────────────
print('⏳ Computing SHAP values (may take 30-60 seconds)...')
explainer   = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

# Handle both 1D (XGBoost) and 2D (RF) SHAP formats
if isinstance(shap_values, list):
    sv = shap_values[1]   # class 1 = Churn
else:
    sv = shap_values

print(f'✅ SHAP values computed for {sv.shape[0]:,} test samples × {sv.shape[1]} features')

In [ ]:
# ── 5.2 SHAP Summary Plot (Global Importance) ────────────────────────────────
print('📊 SHAP Summary Plot — Global Feature Importance\n')
print('How to read: Each dot is one customer.')
print('  X-axis  = SHAP value (impact on churn prediction)')
print('  Color   = Feature value (red=high, blue=low)\n')

plt.figure(figsize=(10, 8))
shap.summary_plot(sv, X_test, show=False, max_display=15,
                  plot_type='dot', color_bar_label='Feature value (normalized)')
plt.title(f'SHAP Summary — {best_model_name}', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_summary.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 5.3 SHAP Bar Plot (Mean Absolute Impact) ─────────────────────────────────
plt.figure(figsize=(10, 6))
shap.summary_plot(sv, X_test, plot_type='bar', show=False, max_display=15)
plt.title('Top 15 Features by Mean |SHAP| Value', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_bar.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── 5.4 SHAP Waterfall Plot — Individual Customer Explanation ────────────────
# Pick a churned customer and explain the prediction step by step
churned_idx   = np.where(y_test.values == 1)[0][:1][0]
retained_idx  = np.where(y_test.values == 0)[0][:1][0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, idx, title, color in zip(
    axes,
    [churned_idx, retained_idx],
    ['Example: Predicted CHURNER', 'Example: Predicted RETAINED'],
    ['#ef4444', '#10b981']
):
    plt.sca(ax)
    exp = shap.Explanation(
        values=sv[idx],
        base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
                    else explainer.expected_value[1],
        data=X_test.iloc[idx].values,
        feature_names=X_test.columns.tolist()
    )
    shap.waterfall_plot(exp, show=False, max_display=10)
    ax.set_title(title, fontweight='bold', color=color, pad=12)

plt.suptitle('Individual Customer Explanations (Waterfall Plots)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('shap_waterfall.png', bbox_inches='tight', dpi=150)
plt.show()
print('\n💡 Each arrow shows how a feature pushes the prediction toward churn (red) or retention (blue)')

In [ ]:
# ── 5.5 SHAP Dependence Plot — Tenure vs Churn ──────────────────────────────
if 'tenure' in X_test.columns:
    plt.figure(figsize=(10, 5))
    shap.dependence_plot(
        'tenure', sv, X_test,
        interaction_index='MonthlyCharges',
        show=False, alpha=0.5
    )
    plt.title('SHAP Dependence: Tenure (colored by Monthly Charges)',
              fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('shap_dependence.png', bbox_inches='tight', dpi=150)
    plt.show()
    print('💡 Customers with LOW tenure + HIGH monthly charges are the highest churn risk!')

---
# 🎛️ Section 6: Interactive Churn Prediction Widget
> Enter a customer's details and get an instant churn probability + explanation

In [ ]:
# ── 6.1 Build the Interactive Widget ─────────────────────────────────────────
FEATURE_COLS = X_train.columns.tolist()

# Style helper
def color_prob(p):
    if p >= 0.7: return '#dc2626', '🔴 HIGH RISK'
    if p >= 0.4: return '#d97706', '🟡 MEDIUM RISK'
    return '#16a34a', '🟢 LOW RISK'

# Widget controls
w_tenure   = widgets.IntSlider(value=12, min=1, max=72, step=1,
                                description='Tenure (mo):', style={'description_width':'140px'},
                                layout=widgets.Layout(width='400px'))
w_monthly  = widgets.FloatSlider(value=70, min=18, max=120, step=1,
                                  description='Monthly Charges:', style={'description_width':'140px'},
                                  layout=widgets.Layout(width='400px'))
w_contract = widgets.Dropdown(options=['Month-to-month','One year','Two year'],
                               value='Month-to-month', description='Contract:',
                               style={'description_width':'140px'},
                               layout=widgets.Layout(width='400px'))
w_internet = widgets.Dropdown(options=['Fiber optic','DSL','No'],
                               value='Fiber optic', description='Internet:',
                               style={'description_width':'140px'},
                               layout=widgets.Layout(width='400px'))
w_security = widgets.ToggleButton(value=False, description='Online Security',
                                   button_style='', layout=widgets.Layout(width='200px'))
w_support  = widgets.ToggleButton(value=False, description='Tech Support',
                                   button_style='', layout=widgets.Layout(width='200px'))
w_senior   = widgets.ToggleButton(value=False, description='Senior Citizen',
                                   button_style='', layout=widgets.Layout(width='200px'))
w_paper    = widgets.ToggleButton(value=True, description='Paperless Billing',
                                   button_style='', layout=widgets.Layout(width='200px'))
w_btn      = widgets.Button(description='🔮 Predict Churn',
                             button_style='primary',
                             layout=widgets.Layout(width='200px', height='40px'))
output     = widgets.Output()

def predict_churn(btn):
    with output:
        output.clear_output(wait=True)
        # Build a baseline row
        sample = pd.DataFrame([{
            col: (X_test[col].median() if X_test[col].dtype != object else X_test[col].mode()[0])
            for col in FEATURE_COLS
        }])

        # Override with widget values (raw, then scale)
        raw_vals = {
            'tenure'        : w_tenure.value,
            'MonthlyCharges': w_monthly.value,
            'TotalCharges'  : w_tenure.value * w_monthly.value,
            'AvgMonthlySpend': w_monthly.value,
            'SeniorCitizen' : int(w_senior.value),
            'PaperlessBilling': int(w_paper.value),
            'ServiceCount'  : 2 + int(w_security.value) + int(w_support.value),
            'NoSupportServices': int(not w_security.value and not w_support.value),
            'IsMonthToMonth': int(w_contract.value == 'Month-to-month'),
            'TenureGroup'   : 0 if w_tenure.value<=12 else (1 if w_tenure.value<=36 else 2),
        }
        for k, v in raw_vals.items():
            if k in sample.columns: sample[k] = v

        # Contract dummies
        for c in ['Contract_One year', 'Contract_Two year']:
            if c in sample.columns: sample[c] = 0
        if w_contract.value == 'One year' and 'Contract_One year' in sample.columns:
            sample['Contract_One year'] = 1
        elif w_contract.value == 'Two year' and 'Contract_Two year' in sample.columns:
            sample['Contract_Two year'] = 1

        # Internet dummies
        for c in ['InternetService_Fiber optic','InternetService_No']:
            if c in sample.columns: sample[c] = 0
        if w_internet.value == 'Fiber optic' and 'InternetService_Fiber optic' in sample.columns:
            sample['InternetService_Fiber optic'] = 1
        elif w_internet.value == 'No' and 'InternetService_No' in sample.columns:
            sample['InternetService_No'] = 1

        # Scale numeric
        num_present = [f for f in num_features if f in sample.columns]
        sample[num_present] = scaler.transform(sample[num_present])

        prob  = best_model.predict_proba(sample)[0][1]
        color, risk_label = color_prob(prob)

        # SHAP for this sample
        sv_s   = explainer.shap_values(sample)
        sv_s1  = sv_s[1][0] if isinstance(sv_s, list) else sv_s[0]
        feat_impact = pd.Series(sv_s1, index=FEATURE_COLS).abs().sort_values(ascending=False)
        top5   = feat_impact.head(5)

        # Display
        display(HTML(f"""
        <div style="font-family:sans-serif; padding:16px; border-radius:12px;
                    border:2px solid {color}; max-width:520px;">
          <h3 style="margin:0 0 8px; color:{color}">{risk_label}</h3>
          <div style="font-size:32px; font-weight:bold; color:{color}; margin-bottom:12px">
            Churn Probability: {prob*100:.1f}%
          </div>
          <div style="background:#f1f5f9; border-radius:8px; padding:10px; margin-bottom:12px">
            <b>📋 Customer Profile Summary</b><br/>
            Tenure: <b>{w_tenure.value} months</b> |
            Monthly: <b>&#36;{w_monthly.value:.0f}</b> |
            Contract: <b>{w_contract.value}</b><br/>
            Internet: <b>{w_internet.value}</b> |
            Security: <b>{'Yes' if w_security.value else 'No'}</b> |
            Support: <b>{'Yes' if w_support.value else 'No'}</b>
          </div>
          <div>
            <b>🔍 Top Factors Driving This Prediction:</b>
            <ol style="margin-top:6px">
              {''.join(f'<li style="margin:4px 0"><code>{feat}</code> (impact={imp:.3f})</li>' for feat, imp in top5.items())}
            </ol>
          </div>
          <div style="margin-top:10px; font-size:12px; color:#6b7280">
            Model: {best_model_name} | Threshold: {best_thresh:.2f} |
            Predicted: {'Churn' if prob >= best_thresh else 'Retain'}
          </div>
        </div>
        """))

w_btn.on_click(predict_churn)

display(HTML('<h3 style="font-family:sans-serif; margin-bottom:4px">🎛️ Customer Churn Predictor</h3>'))
display(HTML('<p style="font-family:sans-serif; color:#6b7280; margin-top:0">Adjust the sliders and click Predict</p>'))
display(widgets.VBox([
    widgets.HBox([w_tenure, w_monthly]),
    widgets.HBox([w_contract, w_internet]),
    widgets.HBox([w_security, w_support, w_senior, w_paper]),
    w_btn,
    output
]))

---
# 💼 Section 7: Business Insights & Recommendations

In [ ]:
# ── 7.1 Churn Risk Segmentation ──────────────────────────────────────────────
# Score the full dataset
X_all        = df_model.drop(columns=['Churn_Binary'])
X_all_scaled = X_all.copy()
X_all_scaled[num_features] = scaler.transform(X_all[num_features])

df_scored = df.copy()
df_scored['ChurnProb'] = best_model.predict_proba(X_all_scaled)[:, 1]
df_scored['RiskSegment'] = pd.cut(
    df_scored['ChurnProb'],
    bins=[0, 0.3, 0.6, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

seg_summary = df_scored.groupby('RiskSegment', observed=True).agg(
    Customers       = ('ChurnProb','count'),
    Avg_Monthly_Charge = ('MonthlyCharges','mean'),
    Avg_Tenure_Months  = ('tenure','mean'),
    Avg_Churn_Prob     = ('ChurnProb','mean')
).round(2)

seg_summary['Est_Revenue_at_Risk'] = (
    seg_summary['Customers'] * seg_summary['Avg_Monthly_Charge'] *
    seg_summary['Avg_Churn_Prob']
).round(0).astype(int)

display(HTML('<h3 style="font-family:sans-serif">📊 Customer Risk Segmentation</h3>'))
display(seg_summary.style.format({'Avg_Churn_Prob':'{:.1%}',
                                   'Avg_Monthly_Charge':'${:.2f}',
                                   'Est_Revenue_at_Risk':'${:,}'}))

In [ ]:
# ── 7.2 Revenue at Risk Visualisation ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Business Impact Analysis', fontsize=14, fontweight='bold')

# Risk segment distribution
ax = axes[0]
seg_counts = df_scored['RiskSegment'].value_counts().reindex(['Low Risk','Medium Risk','High Risk'])
colors_seg = ['#10b981', '#f59e0b', '#ef4444']
bars = ax.bar(seg_counts.index, seg_counts.values, color=colors_seg, edgecolor='white', lw=2)
ax.set_title('Customer Count by Risk Segment', fontweight='bold')
ax.set_ylabel('Number of Customers')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{bar.get_height():,}', ha='center', fontweight='bold')

# Revenue at risk
ax = axes[1]
rev = seg_summary['Est_Revenue_at_Risk']
bars = ax.bar(rev.index, rev.values, color=colors_seg, edgecolor='white', lw=2)
ax.set_title('Estimated Monthly Revenue at Risk', fontweight='bold')
ax.set_ylabel('Revenue at Risk ($)')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f'${x:,.0f}'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'${bar.get_height():,.0f}', ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('business_impact.png', bbox_inches='tight', dpi=150)
plt.show()

total_risk = seg_summary['Est_Revenue_at_Risk'].sum()
high_risk  = seg_summary.loc['High Risk','Est_Revenue_at_Risk']
print(f'\n💰 Total Estimated Monthly Revenue at Risk  : ${total_risk:,.0f}')
print(f'🔴 High-risk segment alone                 : ${high_risk:,.0f}')
print(f'   Retaining just 20% of high-risk customers saves: ${high_risk*0.2:,.0f}/month')

In [ ]:
# ── 7.3 Final Business Recommendations ───────────────────────────────────────
display(HTML("""
<div style="font-family:sans-serif; padding:20px; border-radius:12px;
            background:#f0fdf4; border-left:4px solid #16a34a; max-width:680px">
<h3 style="color:#15803d; margin-top:0">📋 Business Recommendations</h3>
<ol style="line-height:2">
  <li><b>Priority intervention for Month-to-month customers:</b>
      These customers churn at ~42%. Offer a contract upgrade incentive in Month 1-3.</li>
  <li><b>Bundle OnlineSecurity + TechSupport proactively:</b>
      SHAP shows these are top protective features. Include them in onboarding packages.</li>
  <li><b>Early tenure is critical (0-12 months):</b>
      Churn risk is highest in the first year. Deploy a retention campaign at Month 3 and Month 9.</li>
  <li><b>Fiber optic customers with no add-ons:</b>
      Highest monthly charges + lowest service count = high churn risk. Target for upsell.</li>
  <li><b>Deploy model scores in CRM:</b>
      Integrate this model into the CRM to flag accounts with probability &gt; 0.6 for proactive outreach.</li>
</ol>
</div>
"""))

In [ ]:
# ── 7.4 Save Model & Artefacts ───────────────────────────────────────────────
import joblib, os

os.makedirs('churn_model_artefacts', exist_ok=True)
joblib.dump(best_model, 'churn_model_artefacts/best_model.pkl')
joblib.dump(scaler,     'churn_model_artefacts/scaler.pkl')

# Save feature list for later inference
with open('churn_model_artefacts/feature_cols.txt','w') as f:
    f.write('\n'.join(FEATURE_COLS))

print('✅ Model and scaler saved to churn_model_artefacts/')
print(f'   Model   : churn_model_artefacts/best_model.pkl')
print(f'   Scaler  : churn_model_artefacts/scaler.pkl')
print(f'   Features: churn_model_artefacts/feature_cols.txt')

# Optionally download to local machine
# from google.colab import files
# files.download('churn_model_artefacts/best_model.pkl')

In [ ]:
# ── 7.5 Project Summary Card ─────────────────────────────────────────────────
display(HTML(f"""
<div style="font-family:sans-serif; padding:24px; border-radius:12px;
            background:linear-gradient(135deg,#1e3a5f 0%,#1e40af 100%);
            color:white; max-width:680px">
<h2 style="margin:0 0 16px">🏆 Project Complete!</h2>
<table style="width:100%; border-collapse:collapse">
  <tr><td style="padding:6px 0; color:#bfdbfe">Best Model</td>
      <td style="padding:6px 0; font-weight:bold">{best_model_name}</td></tr>
  <tr><td style="padding:6px 0; color:#bfdbfe">Test ROC-AUC</td>
      <td style="padding:6px 0; font-weight:bold">{results[best_model_name]['ROC-AUC']:.4f}</td></tr>
  <tr><td style="padding:6px 0; color:#bfdbfe">Test F1 Score</td>
      <td style="padding:6px 0; font-weight:bold">{results[best_model_name]['F1 Score']:.4f}</td></tr>
  <tr><td style="padding:6px 0; color:#bfdbfe">CV AUC (5-fold)</td>
      <td style="padding:6px 0; font-weight:bold">{results[best_model_name]['CV AUC']:.4f}</td></tr>
  <tr><td style="padding:6px 0; color:#bfdbfe">Optimal Threshold</td>
      <td style="padding:6px 0; font-weight:bold">{best_thresh:.2f}</td></tr>
  <tr><td style="padding:6px 0; color:#bfdbfe">Features Used</td>
      <td style="padding:6px 0; font-weight:bold">{len(FEATURE_COLS)}</td></tr>
  <tr><td style="padding:6px 0; color:#bfdbfe">SMOTE Applied</td>
      <td style="padding:6px 0; font-weight:bold">Yes (training set only)</td></tr>
</table>
<hr style="border-color:#3b82f6; margin:16px 0">
<p style="margin:0; font-size:13px; color:#bfdbfe">
  Artefacts saved: model.pkl · scaler.pkl · 5 plots · interactive widget
</p>
</div>
"""))